# Faz 4 — nnU-Net v2 Eğitimi (Fold 0)

Bu notebook yalnızca eğitim adımını içerir. Ön koşullar:
- `nndet/nifti/` altında NIfTI hacimleri mevcut olmalı
- `nndet/nnunet_raw/Dataset100_Abdomen/` hazır olmalı
- `nndet/nnunet_preprocessed/` plan+preprocess tamamlanmış olmalı

Yukarıdakileri hazırlamak için önce `Faz4_nnDetection_3B_1fold-colab1.ipynb` Section 1–5'i çalıştırın.

In [1]:
!pip install nnunetv2 scipy


  Using cached nnunetv2-2.5.2-py3-none-any.whl
  Using cached acvl_utils-0.2.6-py3-none-any.whl
  Using cached dynamic_network_architectures-0.3.1-py3-none-any.whl
  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
  Using cached imagecodecs-2024.12.30-cp39-cp39-macosx_11_0_arm64.whl.metadata (19 kB)
  Using cached yacs-0.1.8-py3-none-any.whl.metadata (639 bytes)
  Using cached batchgeneratorsv2-0.3.2-py3-none-any.whl.metadata (15 kB)
  Using cached einops-0.8.2-py3-none-any.whl.metadata (13 kB)
INFO: pip is looking at multiple versions of acvl-utils to determine which version is compatible with other requirements. This could take a while.
  Using cached acvl_utils-0.2.5.tar.gz (29 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached acvl_utils-0.2.4.tar.gz (28 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (p

In [2]:
import os, sys, shutil, subprocess, json
import pandas as pd
from pathlib import Path
from typing import List



In [3]:
# Yol ayarları
import shutil, time
IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = not IS_KAGGLE and os.path.exists('/content')

if IS_COLAB:
    NND_ROOT = Path('/content')
else:
    NND_ROOT = Path(os.environ.get('NNUNET_OUT_DIR', r'outputs/nndet'))


LOCAL_PREPROCESSED  = NND_ROOT / "nnunet_preprocessed"
NNUNET_PREPROCESSED = NND_ROOT / "nnunet_preprocessed"
NNUNET_RESULTS      = NND_ROOT / "nnunet_results"

DATASET_ID   = 100
DATASET_NAME = "Abdomen"

os.environ["nnUNet_preprocessed"] = str(NNUNET_PREPROCESSED)
os.environ["nnUNet_results"]      = str(NNUNET_RESULTS)
os.environ["OMP_NUM_THREADS"]     = "1"

In [4]:
# ── Preprocessed veriyi Drive'dan Colab local diske kopyala ───────────────
# Google Drive FUSE üzerinden çok-thread'li NPZ okuma bağlantıyı koparıyor.
# Çözüm: eğitim başlamadan önce preprocessed klasörü /content/ altına al.


if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    drive_pre   = NNUNET_PREPROCESSED
    dataset_sub = f'Dataset{DATASET_ID}_{DATASET_NAME}'
    local_sub   = LOCAL_PREPROCESSED / dataset_sub
    if not local_sub.exists():
        src = drive_pre / dataset_sub
        if not src.exists():
            raise FileNotFoundError(
                f'Preprocessed klasörü bulunamadı: {src}\n'
                'Önce plan_and_preprocess hücresini çalıştırın.'
            )
        print(f'Preprocessed verisi kopyalanıyor: {src}')
        print(f'  → {local_sub}')
        t0 = time.time()
        shutil.copytree(str(src), str(local_sub))
        print(f'  ✓ Tamamlandı ({time.time()-t0:.0f}s)')
    else:
        print(f'Local preprocessed zaten mevcut: {local_sub}')


else:
    print('Colab dışı ortam — kopyalama atlandı.')


Colab dışı ortam — kopyalama atlandı.


In [ ]:
# Ortam değişkenini yerel yola güncelle
os.environ['nnUNet_preprocessed'] = str(LOCAL_PREPROCESSED)
NNUNET_PREPROCESSED = LOCAL_PREPROCESSED
print(f'nnUNet_preprocessed → {os.environ["nnUNet_preprocessed"]}')

In [5]:


import sysconfig as _sc
def _nnunet_cmd(name: str) -> str:
    p = shutil.which(name)
    if p: return p
    for d in [_sc.get_path("scripts"), str(Path(sys.executable).parent),
               "/usr/local/bin", "/opt/conda/bin"]:
        c = Path(d) / name
        if c.exists(): return str(c)
    return name

print(f"nnUNet_preprocessed: {os.environ['nnUNet_preprocessed']}")
print(f"nnUNet_results     : {os.environ['nnUNet_results']}")

_cmd_train = _nnunet_cmd("nnUNetv2_train")
print(f"nnUNetv2_train     : {_cmd_train}  (var={Path(_cmd_train).exists()})")

nnUNet_preprocessed: /Users/ramazanpolat/Desktop/datasets/abdomen/outputs/nndet/nnunet_preprocessed
nnUNet_results     : /Users/ramazanpolat/Desktop/datasets/abdomen/outputs/nndet/nnunet_results
nnUNetv2_train     : /Users/ramazanpolat/Desktop/datasets/abdomen/nndet_venv/bin/nnUNetv2_train  (var=True)


In [6]:
SUPER_CLASSES: List[str] = [
    "acute_cholecystitis",        # 0 → label 1
    "kidney_ureter_stone",        # 1 → label 2
    "acute_pancreatitis",         # 2 → label 3
    "aortic_aneurysm_dissection", # 3 → label 4
    "acute_appendicitis",         # 4 → label 5
    "acute_diverticulitis",       # 5 → label 6
]


## 6. Fold 0 Eğitim

In [7]:
import torch

# ── GPU kontrolü ──────────────────────────────────────────────────────────
_has_cuda = torch.cuda.is_available()
_device   = "cuda" if _has_cuda else "cpu"
print(f"Cihaz: {_device}")

Cihaz: cpu


In [8]:

if _has_cuda:
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

if not _has_cuda:
    print("\n⚠️  CUDA bulunamadı — eğitim GPU gerektirir.")
    print("Bu notebook Colab'da GPU runtime ile çalıştırılmalıdır:")
    print("  Runtime → Change runtime type → T4 GPU (veya A100)")
    raise SystemExit("GPU olmadan eğitim başlatılmadı.")


⚠️  CUDA bulunamadı — eğitim GPU gerektirir.
Bu notebook Colab'da GPU runtime ile çalıştırılmalıdır:
  Runtime → Change runtime type → T4 GPU (veya A100)


SystemExit: GPU olmadan eğitim başlatılmadı.

/Users/ramazanpolat/Desktop/datasets/abdomen/nndet_venv/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3558: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:

# ── nnUNetv2_train ────────────────────────────────────────────────────────
print(f"\nnnU-Net v2 eğitimi — fold 0, 3d_fullres, ~8-15 saat (T4)...")
print(f"Çıktılar: {NNUNET_RESULTS}")


In [ ]:
# import shutil
# import os

# # DİKKAT: Bu komut belirtilen klasörü ve içindeki tüm dosyaları kalıcı olarak siler.
# # Lütfen silmek istediğiniz klasörün doğru olduğundan emin olun.
# folder_to_delete = "/content/nnunet_preprocessed/Dataset100_Abdomen/gt_segmentations" # Burayı silmek istediğiniz klasör yolu ile değiştirin

# if os.path.exists(folder_to_delete):
#     shutil.rmtree(folder_to_delete)
#     print(f"Klasör başarıyla silindi: {folder_to_delete}")
# else:
#     print(f"Klasör bulunamadı: {folder_to_delete}")

In [ ]:
import subprocess
import os

_cmd = _nnunet_cmd("nnUNetv2_train")
# --npz: softmax olasılık haritalarını kaydeder (güven skoru için)
r = subprocess.run(
    [_cmd, str(DATASET_ID), "3d_fullres", "0", "--npz"],
    env={**os.environ}, capture_output=True, text=True
)

print("--- stdout ---")
print(r.stdout)
print("--- stderr ---")
print(r.stderr)

print("Eğitim çıktı kodu:", r.returncode)
if r.returncode != 0:
    print("⚠️  Eğitim hata kodu döndürüyor (bkz. yukarı)")